# Lab 03: Practical Pandas — Sorting, De-duping, GroupBy, Merging, and Column Ops (≈45 min)

**Data file:** `SampleData.csv`

**Goals:** Sorting, de-duplicating, merging, groupby aggregations, and column manipulation.

## 0) Setup & Quick Peek (≈5 min)

In [1]:
import pandas as pd
from pathlib import Path

# Adjust path if needed
csv_path = Path("SampleData.csv")  # or Path("/mnt/data/SampleData.csv")
df = pd.read_csv(csv_path)

df.head()

,Account_no,LocationID,CustomerID,ProductID,Billed_usage_kwh,Invoice_date,Base_charge,Price,Bill
0,1002001,L1001,P01,P01,302,03/05/2020,12.0,0.03,9.06
1,1002001,L1001,P01,P01,520,04/06/2020,12.0,0.03,15.60
2,1002001,L1001,P01,P01,619,05/04/2020,12.0,0.03,18.57
3,1002001,L1001,P01,P01,614,06/03/2020,12.0,0.03,18.42
4,1002001,L1001,P01,P01,1389,07/03/2020,12.0,0.03,41.67


In [2]:
df.shape, df.dtypes

((288, 9),
 Account_no            int64
 LocationID           object
 CustomerID           object
 ProductID            object
 Billed_usage_kwh      int64
 Invoice_date         object
 Base_charge         float64
 Price               float64
 Bill                float64
 dtype: object)

In [3]:
# Convert Invoice_date to datetime and add helpers
df['Invoice_date'] = pd.to_datetime(df['Invoice_date'])
df['Year']  = df['Invoice_date'].dt.year
df['Month'] = df['Invoice_date'].dt.month

df.describe()

,Account_no,Billed_usage_kwh,Invoice_date,Base_charge,Price,Bill,Year,Month
count,2.880000e+02,288.000000,288,288.0,288.000000,288.000000,288.000000,288.000000
mean,1.002012e+06,879.131944,2020-08-12 04:45:00,12.0,0.033333,29.391771,2020.086806,6.774306
min,1.002001e+06,204.000000,2020-03-02 00:00:00,12.0,0.025000,5.100000,2020.000000,1.000000
25%,1.002007e+06,431.750000,2020-05-20 18:00:00,12.0,0.025000,13.680000,2020.000000,4.000000
50%,1.002012e+06,711.500000,2020-08-11 12:00:00,12.0,0.030000,24.905000,2020.000000,7.000000
75%,1.002018e+06,1308.750000,2020-11-04 12:00:00,12.0,0.045000,41.400000,2020.000000,9.250000
max,1.002024e+06,1990.000000,2021-02-01 00:00:00,12.0,0.045000,86.760000,2021.000000,12.000000
std,6.934236e+00,523.463755,NaN,0.0,0.008513,19.433181,0.282040,3.256904


In [4]:
df.shape, df.dtypes

((288, 11),
 Account_no                   int64
 LocationID                  object
 CustomerID                  object
 ProductID                   object
 Billed_usage_kwh             int64
 Invoice_date        datetime64[ns]
 Base_charge                float64
 Price                      float64
 Bill                       float64
 Year                         int32
 Month                        int32
 dtype: object)

In [ ]:
# Convert Account_no to string type
df['Account_no'] = df['Account_no'].astype('string')
df.describe(include=['int64', 'float64'])

,Billed_usage_kwh,Invoice_date,Base_charge,Price,Bill,Year,Month
count,288.000000,288,288.0,288.000000,288.000000,288.000000,288.000000
mean,879.131944,2020-08-12 04:45:00,12.0,0.033333,29.391771,2020.086806,6.774306
min,204.000000,2020-03-02 00:00:00,12.0,0.025000,5.100000,2020.000000,1.000000
25%,431.750000,2020-05-20 18:00:00,12.0,0.025000,13.680000,2020.000000,4.000000
50%,711.500000,2020-08-11 12:00:00,12.0,0.030000,24.905000,2020.000000,7.000000
75%,1308.750000,2020-11-04 12:00:00,12.0,0.045000,41.400000,2020.000000,9.250000
max,1990.000000,2021-02-01 00:00:00,12.0,0.045000,86.760000,2021.000000,12.000000
std,523.463755,NaN,0.0,0.008513,19.433181,0.282040,3.256904


In [8]:
df[['Account_no','LocationID','ProductID']].nunique()

Account_no    24
LocationID    24
ProductID      3
dtype: int64

## Part A — Sorting & De-duping (≈10–12 min)

**A1) Sort by a single column** — See highest usage first.

In [9]:
sorted_usage = df.sort_values(by='Billed_usage_kwh', ascending=False)
sorted_usage[['Account_no','LocationID','ProductID','Invoice_date','Billed_usage_kwh']].head(10)

,Account_no,LocationID,ProductID,Invoice_date,Billed_usage_kwh
113,1002010,L1010,P01,2020-07-29,1990
40,1002004,L1004,P01,2020-07-02,1988
100,1002009,L1009,P03,2020-06-23,1974
208,1002018,L1018,P03,2020-06-29,1961
245,1002021,L1021,P03,2020-07-28,1932
161,1002014,L1014,P02,2020-07-31,1928
78,1002007,L1007,P01,2020-08-25,1926
17,1002002,L1002,P02,2020-07-30,1926
185,1002016,L1016,P01,2020-07-22,1887
281,1002024,L1024,P03,2020-07-23,1875


**A2) Multi-column sort** — Chronological within each account.

In [10]:
sorted_multi = df.sort_values(by=['Account_no','Invoice_date'], ascending=[True, True])
sorted_multi.head(10)

,Account_no,LocationID,CustomerID,ProductID,Billed_usage_kwh,Invoice_date,Base_charge,Price,Bill,Year,Month
0,1002001,L1001,P01,P01,302,2020-03-05,12.0,0.03,9.06,2020,3
1,1002001,L1001,P01,P01,520,2020-04-06,12.0,0.03,15.60,2020,4
2,1002001,L1001,P01,P01,619,2020-05-04,12.0,0.03,18.57,2020,5
3,1002001,L1001,P01,P01,614,2020-06-03,12.0,0.03,18.42,2020,6
4,1002001,L1001,P01,P01,1389,2020-07-03,12.0,0.03,41.67,2020,7
5,1002001,L1001,P01,P01,1335,2020-07-31,12.0,0.03,40.05,2020,7
6,1002001,L1001,P01,P01,1712,2020-08-28,12.0,0.03,51.36,2020,8
7,1002001,L1001,P01,P01,626,2020-09-28,12.0,0.03,18.78,2020,9
8,1002001,L1001,P01,P01,680,2020-10-27,12.0,0.03,20.40,2020,10
9,1002001,L1001,P01,P01,957,2020-11-26,12.0,0.03,28.71,2020,11


**A3) Detect potential duplicates** — Exact vs. key-based duplicates.

In [11]:
# Exact duplicate rows (all columns identical)
first_row = df.iloc[[0]]
new_row = pd.DataFrame({'Account_no': first_row['Account_no'],
                         'CustomerID': 'P99',
                         'ProductID': first_row['ProductID'],
                         'Invoice_date': first_row['Invoice_date']})
df = pd.concat([df, new_row], ignore_index=True)
exact_dups_mask = df.duplicated(keep=False)
df[exact_dups_mask].sort_values(df.columns.tolist()).head()

,Account_no,LocationID,CustomerID,ProductID,Billed_usage_kwh,Invoice_date,Base_charge,Price,Bill,Year,Month


In [12]:
# Business-key duplicates: same account + date + product
key_cols = ['Account_no','Invoice_date','ProductID']
key_dups_mask = df.duplicated(subset=key_cols, keep=False)
df[key_dups_mask].sort_values(key_cols).head()

,Account_no,LocationID,CustomerID,ProductID,Billed_usage_kwh,Invoice_date,Base_charge,Price,Bill,Year,Month
0,1002001,L1001,P01,P01,302.0,2020-03-05,12.0,0.03,9.06,2020.0,3.0
288,1002001,NaN,P99,P01,NaN,2020-03-05,NaN,NaN,NaN,NaN,NaN


**A4) Remove duplicates safely** — Keep the “last” occurrence on key columns.

In [13]:
df_nodup = df.drop_duplicates(subset=key_cols, keep='last').copy()
print("Before:", len(df), "After:", len(df_nodup))

Before: 289 After: 288


## Part B — GroupBy & Aggregations (≈10–12 min)

**B1) Product-level usage and revenue summary**

In [14]:
prod_summary = (
    df_nodup
    .groupby('ProductID', as_index=False)
    .agg(
        total_usage_kwh = ('Billed_usage_kwh','sum'),
        avg_usage_kwh   = ('Billed_usage_kwh','mean'),
        max_usage_kwh   = ('Billed_usage_kwh','max'),
        total_revenue   = ('Bill','sum')
    )
    .sort_values('total_usage_kwh', ascending=False)
)
prod_summary

,ProductID,total_usage_kwh,avg_usage_kwh,max_usage_kwh,total_revenue
1,P02,86349.0,899.468750,1928.0,3885.58
2,P03,85229.0,887.802083,1974.0,2130.89
0,P01,81310.0,855.894737,1990.0,2439.30


**B2) Monthly revenue by product (pivot-style)**

In [15]:
monthly_prod = (
    df_nodup
    .groupby(['Year','Month','ProductID'], as_index=False)
    .agg(monthly_revenue=('Bill','sum'),
         monthly_usage_kwh=('Billed_usage_kwh','sum'))
    .sort_values(['Year','Month','ProductID'])
)
monthly_prod.head(12)

,Year,Month,ProductID,monthly_revenue,monthly_usage_kwh
0,2020.0,3.0,P01,111.81,3727.0
1,2020.0,3.0,P02,224.49,4989.0
2,2020.0,3.0,P03,126.72,5068.0
3,2020.0,4.0,P01,128.88,4296.0
4,2020.0,4.0,P02,243.25,5406.0
5,2020.0,4.0,P03,90.92,3636.0
6,2020.0,5.0,P01,76.08,2536.0
7,2020.0,5.0,P02,165.73,3683.0
8,2020.0,5.0,P03,112.66,4506.0
9,2020.0,6.0,P01,290.25,9675.0


In [16]:
# Filter the monthly summary to a single year (e.g., 2020)
year_filter = 2020
monthly_2020 = monthly_prod[monthly_prod['Year'] == year_filter]

# Find the product with the highest total revenue in that year
top_product_2020 = (
    monthly_2020
    .groupby('ProductID', as_index=False)['monthly_revenue']
    .sum()
    .sort_values('monthly_revenue', ascending=False)
)

print(f"Top product in {year_filter}:")
top_product_2020.head(1)

Top product in 2020:


,ProductID,monthly_revenue
1,P02,3637.35


## Part C — Merging with a Lookup Table (≈8–10 min)

Create a tiny in-memory lookup that adds friendly names and attributes per `ProductID`.

In [17]:
product_lookup = pd.DataFrame({
    'ProductID': ['P01','P02','P03'],
    'ProductName': ['Standard Saver','Peak Flex','Green Choice'],
    'Tier': ['Standard','Peak','Green'],
    'Is_Green': [False, False, True]
})
product_lookup

,ProductID,ProductName,Tier,Is_Green
0,P01,Standard Saver,Standard,False
1,P02,Peak Flex,Peak,False
2,P03,Green Choice,Green,True


**C1) Left-merge onto the fact data**

In [18]:
df_enriched = df_nodup.merge(product_lookup, on='ProductID', how='left')
df_enriched[['ProductID','ProductName','Tier','Is_Green']].drop_duplicates()

,ProductID,ProductName,Tier,Is_Green
0,P01,Standard Saver,Standard,False
11,P02,Peak Flex,Peak,False
23,P03,Green Choice,Green,True


**C2) Re-run a summary with the enriched attributes**

In [19]:
tier_summary = (
    df_enriched
    .groupby('Tier', as_index=False)
    .agg(
        customers=('Account_no','nunique'),
        locations=('LocationID','nunique'),
        total_usage_kwh=('Billed_usage_kwh','sum'),
        total_revenue=('Bill','sum')
    )
    .sort_values('total_revenue', ascending=False)
)
tier_summary

,Tier,customers,locations,total_usage_kwh,total_revenue
1,Peak,8,8,86349.0,3885.58
2,Standard,8,8,81310.0,2439.30
0,Green,8,8,85229.0,2130.89


## Part D — Column Manipulation & Basic Functions (≈8–10 min)

**D1) Create calculated columns** — Recompute bill and compare.

In [20]:
df_enriched['calc_bill'] = df_enriched['Base_charge'] + df_enriched['Price'] * df_enriched['Billed_usage_kwh']
df_enriched['bill_diff'] = (df_enriched['calc_bill'] - df_enriched['Bill']).round(2)

df_enriched['bill_diff'].describe()

count    287.000000
mean      11.999965
std        0.002438
min       11.990000
25%       12.000000
50%       12.000000
75%       12.000000
max       12.010000
Name: bill_diff, dtype: float64

In [21]:
df_enriched.loc[df_enriched['bill_diff'].abs() > 0.01, 
                ['Account_no','Invoice_date','ProductID','Bill','calc_bill','bill_diff']].head()

,Account_no,Invoice_date,ProductID,Bill,calc_bill,bill_diff
0,1002001,2020-04-06,P01,15.60,27.60,12.0
1,1002001,2020-05-04,P01,18.57,30.57,12.0
2,1002001,2020-06-03,P01,18.42,30.42,12.0
3,1002001,2020-07-03,P01,41.67,53.67,12.0
4,1002001,2020-07-31,P01,40.05,52.05,12.0


**D2) Categorize usage with `pd.cut`**

In [22]:
import numpy as np

bins = [0, 500, 1000, 2000, float('inf')]
labels = ['Low','Moderate','High','Very High']
df_enriched['usage_band'] = pd.cut(df_enriched['Billed_usage_kwh'], bins=bins, labels=labels, right=True)
df_enriched['usage_band'].value_counts()

usage_band
High         106
Moderate      94
Low           87
Very High      0
Name: count, dtype: int64

**D3) Add a boolean flag with `np.where`**

In [25]:
df_enriched['is_high_usage'] = np.where(df_enriched['Billed_usage_kwh'] >= 1000, True, False)
df_enriched['is_high_usage'].mean()  # proportion

np.float64(0.3715277777777778)

**D4) Simple row-wise function via `apply`** *(vectorized ops are usually faster)*

In [26]:
def usage_per_dollar(row):
    return row['Billed_usage_kwh'] / row['Bill'] if row['Bill'] else None

df_enriched['kwh_per_dollar'] = df_enriched.apply(usage_per_dollar, axis=1)
df_enriched[['Billed_usage_kwh','Bill','kwh_per_dollar']].head()

,Billed_usage_kwh,Bill,kwh_per_dollar
0,520.0,15.60,33.333333
1,619.0,18.57,33.333333
2,614.0,18.42,33.333333
3,1389.0,41.67,33.333333
4,1335.0,40.05,33.333333


## Part E — Export a Clean Result (≈2–3 min)

In [27]:
monthly_clean = (
    df_enriched
    .groupby(['Year','Month','ProductName'], as_index=False)
    .agg(revenue=('Bill','sum'), usage_kwh=('Billed_usage_kwh','sum'))
    .sort_values(['Year','Month','revenue'], ascending=[True, True, False])
)

monthly_clean.to_csv("monthly_summary.csv", index=False)
print("Wrote monthly_summary.csv")

Wrote monthly_summary.csv


## Optional Stretch

**1) De-dupe simulation** — Append a duplicate row and verify removal.

In [28]:
dupe_row = df_enriched.iloc[[0]]
df_dupe_test = pd.concat([df_enriched, dupe_row], ignore_index=True)
len_before = len(df_dupe_test)
df_dupe_test = df_dupe_test.drop_duplicates(subset=['Account_no','Invoice_date','ProductID'], keep='last')
len_after = len(df_dupe_test)
len_before, len_after

(289, 288)

**2) Multi-metric aggregation**

In [29]:
complex_agg = (
    df_enriched
    .groupby('ProductName', as_index=False)
    .agg(
        revenue_sum=('Bill','sum'),
        bill_min=('Bill','min'),
        bill_max=('Bill','max'),
        accounts=('Account_no','nunique')
    )
    .sort_values('revenue_sum', ascending=False)
)
complex_agg

,ProductName,revenue_sum,bill_min,bill_max,accounts
1,Peak Flex,3885.58,9.54,86.76,8
2,Standard Saver,2439.30,6.21,59.70,8
0,Green Choice,2130.89,5.10,49.35,8
